`Aim:` *Notebook for the purpose of Climate Risk Assessment lecture:* `Download from AWS`

`Author:` Hugo Banderier

# Load CESM2-LE data

For your project, we suggest you work with **CESM2-LE** data. This acronym stands for "Community Earth System Model 2 - Large Ensemble", a project of several universities in the US and in Korea. 

This dataset is the output of the CESM2 model, run 100 times independently, i.e. an ensemble of 100 members in modelling jargon. There are many output variables, on the surface or on 20+ vertical levels, for an output period of 250 years. There are 165 years in the historical period with measured concentrations of CO2 (1850-2015) and 85 years run with CO2 concentrations following the ssp370 scenario.

More details of how these runs are created, that are not very important for us, can be found [on their website](https://www.cesm.ucar.edu/community-projects/lens2).

What's great about it is this enormous amount of data is that *everything* is available online, either on their website with direct downloads or, for a subselection of output variables, on the Amazon AWS servers, a much more convenient way to access it. In this notebook, we demonstrate how to access the latter, and we advise you do the same for your project. 

This notebook should not be used to actually download the data. Instead, modify and use the script `my_download_script.py` that is a condensed and more complete version of this notebook. Here, we demonstrate all the steps so you understand what you are doing. 

In [ ]:
# import packages
import intake
import intake_esm
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import matplotlib.pyplot as plt

Additionally to these packages you need to have `intake-esm`, `aiohttp`, `s3fs` and `fsspec` installed in your environment. If you use the one we provide for you, you already have them. Otherwise, you may uncomment and run the following cell, then **restart the kernel**

In [ ]:
# !uv add xarray fsspec aiohttp intake intake-ESM s3fs cartopy


Here, you can see the list of variables available in CESM2-LE
<https://www2.cesm.ucar.edu/projects/community-projects/LENS2/variable-list/>

In this notebook, we download data that was uploaded on the Amazon servers. Details on this "cloud-optimized" version of the dataset can be found [here](https://ncar.github.io/cesm2-le-aws/model_documentation.html).

In [ ]:
## Configure
var = "RAIN"
outpath = Path("/Users/bandelol/Documents/code_local/data/cesm-test")
## change this!

col_url = (
    "https://ncar-cesm2-lens.s3-us-west-2.amazonaws.com/catalogs/aws-cesm2-le.json"
)
catalog = intake.open_esm_datastore(col_url)

This `.json` file informs us on the content of what is stored in this big data repository. The `intake` package stores it in a weird format that is not super readable to us, but we can turn it into the much more readable format of pandas dataframe.

It also contains the paths to the actual data files, and the `intake` package can use them to give us `xarray` datasets with this data. 

In [ ]:
catalog.df

And this is a comprehensive list of all variables available through this method. If you have a specific need for a CESM2 variable not in this list, we can help you download it the classic way but be aware that it is significantly more work, and a lot can be done with the variables in this list.

In [ ]:
catalog.df[["variable", "long_name", "vertical_levels", "units", "spatial_domain"]].drop_duplicates()

We start by searching for the data we actually want, using the `search` function. In this example, we use daily mean rain using the CMIP6 forcing.

In [ ]:
catalog_subset = catalog.search(variable=var, frequency='daily', forcing_variant="cmip6")
catalog_subset.df

We can finally extract two datasets, one per period, using `intake`. At this point, the datasets are not loaded in memory (they are way too large) but represent links to remote data. If you see warning messages about chunking, there is nothing you can do about it, sorry.

In [ ]:
dsets = catalog_subset.to_dataset_dict(storage_options={'anon':True})

In [ ]:
ds_past = dsets["lnd.historical.daily.cmip6"]
ds_future = dsets["lnd.ssp370.daily.cmip6"]
ds_past

A great feature of `xarray` Datasets is that we can perform our subselection (time and region that interest us) before loading the data in memory, making using large data possible. We will select 30 years in the historical run and 30 years in the ssp370 run, a few members each only, and the region that we are interested in: more or less Switzerland.

In [ ]:
import cartopy.crs as ccrs
from cartopy.feature import COASTLINE
from matplotlib.patches import Rectangle
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent([-22, 40, 31, 72])
ax.add_feature(COASTLINE)
minlon, maxlon, minlat, maxlat = 6.02260949059, 10.4427014502, 45.7769477403, 47.8308275417
ax.add_patch(Rectangle([minlon, minlat], maxlon - minlon, maxlat - minlat, facecolor="none", edgecolor="red", linewidth=2))

In [ ]:
def standardize(da: xr.Dataset | xr.DataArray) -> xr.Dataset | xr.DataArray:
    if (da.lon.max() > 180) and (da.lon.min() >= 0): # we prefer longitude to go from -180 to +180 rather than 0 to 360
        da = da.assign_coords(lon=(((da.lon + 180) % 360) - 180))
        da = da.sortby("lon")
    if np.diff(da.lat.values)[0] < 0: # and increasing latitudes
        da = da.reindex(lat=da.lat[::-1])
    return da

In [ ]:
minlon, maxlon, minlat, maxlat = -30, 40, 20, 80 # download more than you need, just in case
ds_past_ns = (
    standardize(ds_past)
    .isel(member_id=np.arange(5))
    .isel(time=np.isin(ds_past.time.dt.year, np.arange(1970, 2000)))
    .sel(lon=slice(minlon, maxlon))
    .sel(lat=slice(minlat, maxlat))
)
ds_future_ns = (
    standardize(ds_future)
    .isel(member_id=np.arange(5))
    .isel(time=np.isin(ds_future.time.dt.year, np.arange(2070, 2100)))
    .sel(lon=slice(minlon, maxlon))
    .sel(lat=slice(minlat, maxlat))
)
ds_future_ns

We can finally load this very manageable amount of data into memory. 

This is when the actual downloading happens, and it might be very slow. 

This is not done very optimally by xarray, **especially for data that is originally 3D**, like wind or atmospheric temperature, even if you only ask for a 2D slice, because of the way the data is *chunked* in the remote storage. Unfortunately there is nothing we can do about this except wait.

**For your project, you should therefore do it as early as possible**

For your project, you should really run this as a standalone python script, using the `nohup` command to run it in the background for a long time.

If you are having problems accessing the data that you need in a reasonable time, there are other more involved options we can explain to you, but remember that **your project does not need to involve super heavy data to be good**!

```bash
$ nohup python my_download_script.py &
```

In [ ]:
from dask.diagnostics import ProgressBar
with ProgressBar():
    ds_past_ns = ds_past_ns.load()
ds_past_ns.to_netcdf(outpath.joinpath("precip_past.nc"))
del ds_past_ns
with ProgressBar():
    ds_future_ns = ds_future_ns.load()
ds_future_ns.to_netcdf(outpath.joinpath("precip_future.nc"))
del ds_future_ns

In [ ]:
ds_future_ns = xr.open_dataset(outpath.joinpath("precip_future.nc"))
ds_future_ns

For today, we simply give you this example data so you can try the other notebook. We also provide an example downloading script that you can adapt and run yourself when comes the time to download data for your project!

### Bonus: Read in the Grid Data

We also have Zarr stores of grid data, such as the area of each grid cell. We will need to follow a similar process here, querying for our experiment and extracting the dataset

In [ ]:
grid_subset = catalog.search(component='atm', frequency='static', experiment='historical', forcing_variant='cmip6')

We can load in the dataset, calling `pop_item`, which grabs the dataset, assigning `_` to the meaningless key

In [ ]:
_, grid = grid_subset.to_dataset_dict(aggregate=False, zarr_kwargs={"consolidated": True}, storage_options={'anon':True}).popitem()
grid